# Synthetic Product Dataset Generation

This notebook generates a synthetic built-in oven product portfolio using predifined product characteristics, generation probabilities, and configuration rules.

## Objective

The goal is to generate valid and realistic synthetic products rather than using unconstrained random combinations.

Generation logic:

1. Read master data from excel
2. Validate source tables
3. Sample driver characteristics
4. Apply configuration rules
5. Reject incompatible configurations
6. Generate valid products
7. Export PRODUCT_MASTER

In [1]:
import pandas as pd
import numpy as np

In [2]:
master_file = "../data/raw/oven_configurator_master_v1.0.xlsx"
excel_file = pd.ExcelFile(master_file)
excel_file.sheet_names

['Definition',
 'Proje Tanımlama',
 'PROJE TANIM',
 'Örnek Veri Seti',
 'Data Types',
 'DATASET_INFO',
 'README',
 'SYNTHETIC_GENERATION',
 'Dataset Architecture v1.0',
 'CHARACTERISTIC_DICTIONARY',
 'CHARACTERISTIC_VALUES',
 'CONFIGURATION_RULES',
 'PRODUCT_MASTER',
 'OVEN_FUNCTION_SET',
 'CUSTOMER_REQUEST',
 'REQUEST_GROUND_TRUTH']

In [3]:
characteristic_dictionary = pd.read_excel(
    master_file,
    sheet_name="CHARACTERISTIC_DICTIONARY", #pythonda sayma 0'dan başlar, notların bulunduğu ilk satırı hariç tutmak için header=1 kullanıyoruz
    header=1
)

characteristic_dictionary.head()



,CHARACTERISTIC,DATA_TYPE,ALLOWED_VALUES / RANGE,MATCH_METHOD,UNIT,DESCRIPTION
0,WIDTH_CM,NUMERIC,"45, 60, 70, 90",EXACT,cm,Product installation width
1,PACKAGING_TYPE,CATEGORICAL,"SHRINK, BOX",EXACT,-,Design of product packaging
2,PANEL_TYPE,CATEGORICAL,"GLASS, METAL",EXACT,-,Control panel material
3,PANEL_COLOR,CATEGORICAL,"BLACK, WHITE, SILVER, STAINLESS STEEL, DARK SI...",EXACT,-,Control panel color
4,POWER_CABLE,CATEGORICAL,"W/O CABLE, W/O PLUG, EU TYPE PLUG, US TYPE PLU...",EXACT,-,Power cable availability of product based on p...


In [4]:
print("shape:", characteristic_dictionary.shape)    #DataFrame boyutunu gösterir (satır sayısı, sütun sayısı)

print("\nColumns:")
print(characteristic_dictionary.columns.tolist())   #DataFrame sütun isimlerini liste olarak gösterir

print("\nDataFrame Info:")
characteristic_dictionary.info()                    #DataFrame hakkında genel bilgi verir

print("nMissing values:")
print(characteristic_dictionary.isna().sum())       #Her sütundaki eksik değer sayısını gösterir

shape: (24, 6)

Columns:
['CHARACTERISTIC', 'DATA_TYPE', 'ALLOWED_VALUES / RANGE', 'MATCH_METHOD', 'UNIT', 'DESCRIPTION']

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   CHARACTERISTIC          24 non-null     object
 1   DATA_TYPE               24 non-null     object
 2   ALLOWED_VALUES / RANGE  24 non-null     object
 3   MATCH_METHOD            24 non-null     object
 4   UNIT                    24 non-null     object
 5   DESCRIPTION             24 non-null     object
dtypes: object(6)
memory usage: 1.3+ KB
nMissing values:
CHARACTERISTIC            0
DATA_TYPE                 0
ALLOWED_VALUES / RANGE    0
MATCH_METHOD              0
UNIT                      0
DESCRIPTION               0
dtype: int64


## Load and Validate Characteristic Values

The CHARACTERISTIC_VALUES table defines the valid values associated with each product characteristic.

This table will be validated aggainst the CHARACTERISTIC_DICTIONARY to ensure consistency between characteristic definitions and their available values.

In [5]:
characteristic_values = pd.read_excel(
    master_file,
    sheet_name="CHARACTERISTIC_VALUES",
    header=0
)

characteristic_values.head()

,CHARACTERISTIC,VALUE,DEFINITION
0,WIDTH_CM,V001,45
1,WIDTH_CM,V002,60
2,WIDTH_CM,V003,70
3,WIDTH_CM,V004,90
4,PACKAGING_TYPE,V001,SHRINK


In [6]:
print("Shape:", characteristic_values.shape)    #DataFrame boyutunu gösterir (satır sayısı, sütun sayısı)

print("\nColumns:")
print(characteristic_values.columns.tolist())   #DataFrame sütun isimlerini liste olarak gösterir

print("\nDataFrame Info:")
characteristic_values.info()                    #DataFrame hakkında genel bilgi verir

print("\nMissing Values:")
print(characteristic_values.isna().sum())       #Her sütundaki eksik değer sayısını gösterir

print("\nDuplicate Rows:")
print(characteristic_values.duplicated().sum()) #Her sütundaki yinelenen satır sayısını gösterir

Shape: (90, 3)

Columns:
['CHARACTERISTIC', 'VALUE', 'DEFINITION']

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90 entries, 0 to 89
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   CHARACTERISTIC  90 non-null     object
 1   VALUE           90 non-null     object
 2   DEFINITION      90 non-null     object
dtypes: object(3)
memory usage: 2.2+ KB

Missing Values:
CHARACTERISTIC    0
VALUE             0
DEFINITION        0
dtype: int64

Duplicate Rows:
0


In [7]:
dictionary_characteristics = set(
    characteristic_dictionary["CHARACTERISTIC"]         #Set of unique characteristics from the characteristic dictionary
)

values_characteristics = set(
    characteristic_values["CHARACTERISTIC"]             #Set of unique characteristics from the characteristic values
)

undefined_characteristics = (
    values_characteristics - dictionary_characteristics #Set of characteristics that are in the values but not in the dictionary
)

print("Characteristics in Dictionary:", len(dictionary_characteristics))    #Set of unique characteristics from the characteristic dictionary
print("Characteristics in Values:", len(values_characteristics))            #Set of unique characteristics from the characteristic values

print("\nUndefined Characteristics:")
print(undefined_characteristics)            #Set of characteristics that are in the values but not in the dictionary

missing_value_definitions = (
    dictionary_characteristics - values_characteristics #Set of characteristics that are in the dictionary but not in the values
)

print("\nCharacteristics Without Value Definitions:")
print(missing_value_definitions)        #Set of characteristics that are in the dictionary but not in the values

Characteristics in Dictionary: 24
Characteristics in Values: 24

Undefined Characteristics:
set()

Characteristics Without Value Definitions:
set()


## Load and Validate Configuration Rules

The CONFIGURATION_RULES table defines the domain constraints used to generate valid synthetic product configurations.

The rules will be validated against the characteristic and value master tables before they are used by the synthetic product generator.

In [8]:
configuration_rules = pd.read_excel(
    master_file,
    sheet_name="CONFIGURATION_RULES"
)

configuration_rules.head(10)

,RuleId,Type,Operator,CHARACTERISTIC,VALUE,DEFINITION
0,1,Auto Select,Action,TURNSPIT,V001,YES
1,1,AND,Rule,OVEN_FUNCTIONS,V004,04-OvenLight | Defrost | Top+Bottom | Top+Bott...
2,1,AND,Rule,OVEN_FUNCTIONS,V007,07-Defrost | Fan+Ring | Top+Bottom | Top+Bott...
3,2,Auto Select,Action,TURNSPIT,V002,NO
4,2,AND,Rule,OVEN_FUNCTIONS,V001,01-Top+Bottom | Bottom | Top
5,2,AND,Rule,OVEN_FUNCTIONS,V002,02-Defrost | Top+Bottom | Top+Bottom+Fan | Gri...
6,2,AND,Rule,OVEN_FUNCTIONS,V003,03-Bottom | Top | Top+Bottom | Top+Bottom+Fan ...
7,2,AND,Rule,OVEN_FUNCTIONS,V005,05-OvenLight | Defrost | Top+Bottom | Top+Bott...
8,2,AND,Rule,OVEN_FUNCTIONS,V006,06-Defrost | Fan+Ring | Top+Bottom | Top+Bott...
9,2,AND,Rule,OVEN_FUNCTIONS,V008,08-OvenLight | Defrost | Fan+Ring | Top+Bottom...


### Validate Rule Characterisic-Value Pairs

Each characteristic-value pair used in CONFIGURATION_RULES must exist in CHARACTERISTIC_VALUES.

This validation prevents configuration rules from referencing undefined characteristics or invalid values.

In [9]:
# Create valid (CHARACTERISTIC, VALUE) pairs from the master value table
valid_pairs = set(
    zip(
        characteristic_values["CHARACTERISTIC"],
        characteristic_values["VALUE"]
    )
)

# Create the pairs referenced by the configuration rules
rule_pairs = set(
    zip(
        configuration_rules["CHARACTERISTIC"],
        configuration_rules["VALUE"]
    )
)

# Find rule pairs that do not exist in CHARACTERISTIC_VALUES
invalid_rule_pairs = rule_pairs - valid_pairs

print("Valid characteristic-value pairs:", len(valid_pairs))
print("Characteristic-value pairs used in rules:", len(rule_pairs))

print("\nInvalid rule pairs:")
print(invalid_rule_pairs)

Valid characteristic-value pairs: 90
Characteristic-value pairs used in rules: 72

Invalid rule pairs:
set()


### Validate Configuration Rule Structure

The configuration rules are grouped by RuleId.

Each rule group may contain:
- condition rows (`Operator = Rule`)
- action rows (`Operator = Action`)

The purpose of this validation is to inspect whether each RuleId has the expected structure before implementing the rule engine.

In [10]:
# General structure of the rule table
print("Shape:", configuration_rules.shape)

print("\nNumber of unique RuleIds:")
print(configuration_rules["RuleId"].nunique())          #Number of unique RuleIds in the configuration rules

print("\nRule types:")
print(configuration_rules["Type"].value_counts())       #Count of each unique rule type in the configuration rules

print("\nOperators:")
print(configuration_rules["Operator"].value_counts())    #Count of each unique operator in the configuration rules

Shape: (170, 6)

Number of unique RuleIds:
37

Rule types:
Type
AND            116
Enable          21
Auto Select     20
INC             11
Disable          2
Name: count, dtype: int64

Operators:
Operator
Rule               116
Action              43
Incompatibility     11
Name: count, dtype: int64


### Inspect Rule Groups

Each RuleId represents a configuration rule group.

This step summarizes the internal structure of each rule before implementing the rule engine.

In [11]:
rule_summary = (
    configuration_rules
    .groupby("RuleId")
    .agg(
        ROW_COUNT=("RuleId", "size"),                                   #Count of rows for each RuleId
        RULE_ROWS=("Operator", lambda x: (x == "Rule").sum()),          #Count of rows with operator "Rule" for each RuleId
        ACTION_ROWS=("Operator", lambda x: (x == "Action").sum()),      #Count of rows with operator "Action" for each RuleId
        INC_ROWS=("Operator", lambda x: (x == "Incompatibility").sum()) #Count of rows with operator "Incompatibility" for each RuleId
    )
)

rule_summary

,ROW_COUNT,RULE_ROWS,ACTION_ROWS,INC_ROWS
RuleId,,,,
1,3,2,1,0
2,11,10,1,0
3,12,11,1,0
4,3,1,2,0
5,2,1,1,0
6,3,2,1,0
7,5,4,1,0
8,3,2,1,0
9,4,3,1,0


### Validate Rule Group Consistency

A RuleId must follow one of two valid structures:

1. Condition-Action rule:
   - one or more Rule rows
   - one or more Action rows
   - no Incompatibility rows

2. Incompatibility rule:
   - one or more Incompatibility rows
   - no Rule or Action rows

This validation ensures that incompatible rule structures are not mixed within the same RuleId.

In [12]:
# Identify RuleIds that mix incompatibility rows with normal Rule/Action rows

mixed_rule_groups = rule_summary[
    (rule_summary["INC_ROWS"] > 0)
    &
    (
        (rule_summary["RULE_ROWS"] > 0)
        | #OR
        (rule_summary["ACTION_ROWS"] > 0)
    )
]

print("Mixed rule groups:")
print(mixed_rule_groups)

Mixed rule groups:
Empty DataFrame
Columns: [ROW_COUNT, RULE_ROWS, ACTION_ROWS, INC_ROWS]
Index: []


### Validate Condition-Action Completeness

Every non-incompatibility rule group must contain:

- at least one condition row (`Operator = Rule`)
- at least one action row (`Operator = Action`)

This validation detects incomplete configuration rules before the rule engine is implemented.

In [13]:
# Select normal rule groups (non-incompatibility rules)
normal_rule_groups = rule_summary[
    rule_summary["INC_ROWS"] == 0
]

# Find normal rules with missing conditions or missing actions
incomplete_rule_groups = normal_rule_groups[
    (normal_rule_groups["RULE_ROWS"] == 0)
    |  #OR                                             
    (normal_rule_groups["ACTION_ROWS"] == 0)
]

print("Incomplete normal rule groups:")
print(incomplete_rule_groups)

Incomplete normal rule groups:
Empty DataFrame
Columns: [ROW_COUNT, RULE_ROWS, ACTION_ROWS, INC_ROWS]
Index: []


## Identify Driver and Rule-Controlled Characteristics

Before generating synthetic products, characteristics must be separated according to their role in the configuration system.

- **Driver characteristics:** selected during product generation and may trigger configuration rules.
- **Auto-selected characteristics:** deterministically assigned by Auto Select rules.
- **Enabled/Disabled characteristics:** their available values are constrained by rules.
- **Incompatibility characteristics:** participate in combinations that must be validated.

This separation prevents dependent characteristics from being sampled independently.

In [14]:
# Characteristics used as rule conditions
condition_characteristics = set(                                #set
    configuration_rules.loc[                                    #loc function is used to access a group of rows and columns by labels or a boolean array
        configuration_rules["Operator"] == "Rule",
        "CHARACTERISTIC"
    ]
)

# Characteristics controlled by actions
action_characteristics = set(                                   #set of unique characteristics from the
    configuration_rules.loc[                                    #loc function is used to access a group of rows and columns by labels or a boolean array
        configuration_rules["Operator"] == "Action",
        "CHARACTERISTIC"
    ]
)

# Characteristics participating in incompatibility rules
incompatibility_characteristics = set(                          #set of unique characteristics from the configuration rules with operator "Incompatibility"
    configuration_rules.loc[                                    #loc function is used to access a group of rows and columns by labels or a boolean array
        configuration_rules["Operator"] == "Incompatibility",
        "CHARACTERISTIC"
    ]
)

print("Condition characteristics:")
print(sorted(condition_characteristics))        #sorted by alphabetical order

print("\nAction-controlled characteristics:")
print(sorted(action_characteristics))

print("\nIncompatibility characteristics:")
print(sorted(incompatibility_characteristics))

Condition characteristics:
['HANDLE_TYPE', 'KNOB_TYPE', 'OVEN_CONTROL', 'OVEN_FUNCTIONS', 'PANEL_COLOR', 'WIDTH_CM']

Action-controlled characteristics:
['CLEANING', 'ENERGY_CLASS', 'HANDLE_COLOR', 'KNOB_COLOR', 'KNOB_STRUCTURE', 'NUMBER_OF_FUNCTIONS', 'OVEN_CONTROL', 'OVEN_TYPE', 'OVEN_VOLUME_L', 'PANEL_TYPE', 'PIZZA_TRAY', 'TURNSPIT', 'WIFI']

Incompatibility characteristics:
['BAKING_TRAY', 'HANDLE_COLOR', 'KNOB_COLOR', 'ROASTING_TRAY', 'WIRE_SHELF']


## Build Characteristic Role Matrix

A characteristic may participate in configuration logic in more than one role.

For each characteristic, we identify whether it is used as:

- **CONDITION**: used as an input condition in a rule
- **ACTION**: controlled or assigned by a rule action
- **INCOMPATIBILITY**: participates in an incompatible combination

A characteristic may have one, two, or all three roles.
This role matrix will help us understand the dependency structure before defining
driver and rule-controlled characteristics for synthetic product generation.

In [15]:
# Start with all characteristics defined in the dictionary
role_matrix = pd.DataFrame({
    "CHARACTERISTIC": sorted(dictionary_characteristics)
})

# Check whether each characteristic appears in each rule role
role_matrix["IS_CONDITION"] = role_matrix["CHARACTERISTIC"].isin(
    condition_characteristics
)

role_matrix["IS_ACTION"] = role_matrix["CHARACTERISTIC"].isin(
    action_characteristics
)

role_matrix["IS_INCOMPATIBILITY"] = role_matrix["CHARACTERISTIC"].isin(
    incompatibility_characteristics
)

role_matrix

,CHARACTERISTIC,IS_CONDITION,IS_ACTION,IS_INCOMPATIBILITY
0,BAKING_TRAY,False,False,True
1,CLEANING,False,True,False
2,ENERGY_CLASS,False,True,False
3,HANDLE_COLOR,False,True,True
4,HANDLE_TYPE,True,False,False
5,KNOB_COLOR,False,True,True
6,KNOB_STRUCTURE,False,True,False
7,KNOB_TYPE,True,False,False
8,NUMBER_OF_FUNCTIONS,False,True,False
9,OVEN_CONTROL,True,True,False


In [16]:
def get_roles(row):                         #define a function to get the roles of a characteristic based on the boolean columns in the role_matrix
    roles = []                              #function cerates an empty list called roles to store the roles of a characteristic based on the boolean columns in the role_matrix

    if row["IS_CONDITION"]:
        roles.append("CONDITION")           #append "CONDITION" to the roles list if the characteristic is a condition

    if row["IS_ACTION"]:
        roles.append("ACTION")              #append "ACTION" to the roles list if the characteristic is an action

    if row["IS_INCOMPATIBILITY"]:
        roles.append("INCOMPATIBILITY")     #append "INCOMPATIBILITY" to the roles list if the characteristic is an incompatibility

    if not roles:
        return "NONE"

    return " + ".join(roles)                #Join the roles in the list with " + " as a separator and return the resulting string


role_matrix["ROLE"] = role_matrix.apply(get_roles, axis=1)  #apply the get_roles function to each row of the role_matrix DataFrame and create a new column "ROLE" with the resulting role strings
                                                            #axis=1 means that the function is applied to each row (as opposed to each column)
role_matrix

,CHARACTERISTIC,IS_CONDITION,IS_ACTION,IS_INCOMPATIBILITY,ROLE
0,BAKING_TRAY,False,False,True,INCOMPATIBILITY
1,CLEANING,False,True,False,ACTION
2,ENERGY_CLASS,False,True,False,ACTION
3,HANDLE_COLOR,False,True,True,ACTION + INCOMPATIBILITY
4,HANDLE_TYPE,True,False,False,CONDITION
5,KNOB_COLOR,False,True,True,ACTION + INCOMPATIBILITY
6,KNOB_STRUCTURE,False,True,False,ACTION
7,KNOB_TYPE,True,False,False,CONDITION
8,NUMBER_OF_FUNCTIONS,False,True,False,ACTION
9,OVEN_CONTROL,True,True,False,CONDITION + ACTION


In [17]:
def classify_generation_role(row):

    if row["IS_ACTION"] and row["IS_CONDITION"]:
        return "INTERMEDIATE"

    elif row["IS_ACTION"]:
        return "DERIVED"

    elif row["IS_CONDITION"]:
        return "DRIVER"

    elif row["IS_INCOMPATIBILITY"]:
        return "VALIDATION_ONLY"

    else:
        return "INDEPENDENT"


role_matrix["INFERRED_GENERATION_ROLE"] = role_matrix.apply(     #INFERRED_GENERATION_ROLE column is created by applying the classify_generation_role function to each row of the role_matrix DataFrame
    classify_generation_role,
    axis=1
)

role_matrix[
    ["CHARACTERISTIC", "ROLE", "INFERRED_GENERATION_ROLE"]
]

,CHARACTERISTIC,ROLE,INFERRED_GENERATION_ROLE
0,BAKING_TRAY,INCOMPATIBILITY,VALIDATION_ONLY
1,CLEANING,ACTION,DERIVED
2,ENERGY_CLASS,ACTION,DERIVED
3,HANDLE_COLOR,ACTION + INCOMPATIBILITY,DERIVED
4,HANDLE_TYPE,CONDITION,DRIVER
5,KNOB_COLOR,ACTION + INCOMPATIBILITY,DERIVED
6,KNOB_STRUCTURE,ACTION,DERIVED
7,KNOB_TYPE,CONDITION,DRIVER
8,NUMBER_OF_FUNCTIONS,ACTION,DERIVED
9,OVEN_CONTROL,CONDITION + ACTION,INTERMEDIATE


## Build Characteristic Dependency Map

A dependency exists when a characteristic used as a rule condition controls
a characteristic used as an action within the same RuleId.

The dependency map is derived automatically from CONFIGURATION_RULES rather
than being manually defined.

This helps determine the correct generation order for synthetic products and
makes the architecture adaptable when configuration rules change.

In [18]:
dependency_pairs = []

for rule_id, group in configuration_rules.groupby("RuleId"):    #group the configuration_rules DataFrame by "RuleId" and iterate over each group, where rule_id is the unique RuleId and group is the corresponding subset of the DataFrame for that RuleId

    conditions = set(                                   #set of unique characteristics from the group where the operator is "Rule"
        group.loc[                                      #loc function is used to access a group of rows and columns by labels or a boolean array
            group["Operator"] == "Rule",
            "CHARACTERISTIC"
        ]
    )

    actions = set(                                      #set of unique characteristics from the group where the operator is "Action" 
        group.loc[                                      #loc function is used to access a group of rows and columns by labels or a boolean array  
            group["Operator"] == "Action",
            "CHARACTERISTIC"
        ]
    )

    for condition in conditions:
        for action in actions:
            dependency_pairs.append(
                {
                    "RuleId": rule_id,
                    "FROM_CHARACTERISTIC": condition,
                    "TO_CHARACTERISTIC": action
                }
            )

dependency_df = (
    pd.DataFrame(dependency_pairs)
    .drop_duplicates()
    .sort_values(
        ["FROM_CHARACTERISTIC", "TO_CHARACTERISTIC", "RuleId"]
    )
    .reset_index(drop=True)     #Reset the index of the DataFrame after sorting, and drop the old index to avoid adding it as a new column. drop=True means that the old index is not added as a new column in the DataFrame
)

dependency_df

,RuleId,FROM_CHARACTERISTIC,TO_CHARACTERISTIC
0,17,HANDLE_TYPE,HANDLE_COLOR
1,18,HANDLE_TYPE,HANDLE_COLOR
2,15,KNOB_TYPE,KNOB_COLOR
3,16,KNOB_TYPE,KNOB_COLOR
4,13,KNOB_TYPE,KNOB_STRUCTURE
5,14,KNOB_TYPE,KNOB_STRUCTURE
6,24,OVEN_CONTROL,ENERGY_CLASS
7,36,OVEN_CONTROL,PANEL_TYPE
8,37,OVEN_CONTROL,PANEL_TYPE
9,34,OVEN_CONTROL,WIFI


In [19]:
characteristic_dependencies = (
    dependency_df[          #define a new DataFrame characteristic_dependencies that contains only the unique pairs of FROM_CHARACTERISTIC and TO_CHARACTERISTIC from the dependency_df DataFrame
        ["FROM_CHARACTERISTIC", "TO_CHARACTERISTIC"]
    ]
    .drop_duplicates()
    .sort_values(
        ["FROM_CHARACTERISTIC", "TO_CHARACTERISTIC"]
    )
    .reset_index(drop=True)
)

print(
    "Unique characteristic dependencies:",
    len(characteristic_dependencies)
)

characteristic_dependencies

Unique characteristic dependencies: 16


,FROM_CHARACTERISTIC,TO_CHARACTERISTIC
0,HANDLE_TYPE,HANDLE_COLOR
1,KNOB_TYPE,KNOB_COLOR
2,KNOB_TYPE,KNOB_STRUCTURE
3,OVEN_CONTROL,ENERGY_CLASS
4,OVEN_CONTROL,PANEL_TYPE
5,OVEN_CONTROL,WIFI
6,OVEN_FUNCTIONS,CLEANING
7,OVEN_FUNCTIONS,ENERGY_CLASS
8,OVEN_FUNCTIONS,NUMBER_OF_FUNCTIONS
9,OVEN_FUNCTIONS,OVEN_CONTROL


## Determine Characteristic Generation Levels

The dependency map is used to determine the order in which characteristics
should be generated.

Characteristics with no parent dependencies can be generated first.
Dependent characteristics are generated only after their parent
characteristics are available.

In [20]:
# Characteristics that appear as dependency sources
parent_characteristics = set(
    characteristic_dependencies["FROM_CHARACTERISTIC"]
)

# Characteristics that appear as dependency targets
child_characteristics = set(
    characteristic_dependencies["TO_CHARACTERISTIC"]
)

print("Parent characteristics:")
print(sorted(parent_characteristics))

print("\nChild characteristics:")
print(sorted(child_characteristics))

Parent characteristics:
['HANDLE_TYPE', 'KNOB_TYPE', 'OVEN_CONTROL', 'OVEN_FUNCTIONS', 'PANEL_COLOR', 'WIDTH_CM']

Child characteristics:
['CLEANING', 'ENERGY_CLASS', 'HANDLE_COLOR', 'KNOB_COLOR', 'KNOB_STRUCTURE', 'NUMBER_OF_FUNCTIONS', 'OVEN_CONTROL', 'OVEN_TYPE', 'OVEN_VOLUME_L', 'PANEL_TYPE', 'PIZZA_TRAY', 'TURNSPIT', 'WIFI']


In [21]:
root_characteristics = (
    parent_characteristics - child_characteristics  #- means set difference, so this line calculates the set of characteristics that are parents but not children, which are considered root characteristics
)

print("Root characteristics:")
print(sorted(root_characteristics))

Root characteristics:
['HANDLE_TYPE', 'KNOB_TYPE', 'OVEN_FUNCTIONS', 'PANEL_COLOR', 'WIDTH_CM']


### Calculate Dependency Levels

Characteristics are assigned to dependency levels based on the rule graph.

- Level 0 contains root characteristics.
- A characteristic can be assigned to the next level only after all of its parent characteristics have already been assigned.
- This creates a valid generation order for dependent product characteristics.

In [22]:
# Start with root characteristics at Level 0
dependency_levels = {
    characteristic: 0
    for characteristic in root_characteristics
}

remaining_characteristics = (
    child_characteristics - set(dependency_levels.keys()) # keys() returns a view of the keys in the dependency_levels dictionary, which are the characteristics that have already been assigned a level. The set() function converts this view into a set, allowing for set operations. The difference operation (-) is then used to find characteristics that are in child_characteristics but not in the set of already assigned characteristics, resulting in a set of characteristics that still need to be assigned a level
)

current_level = 1

while remaining_characteristics:    #as long as there are characteristics that have not yet been assigned a level, the while loop continues to iterate. The loop will terminate when all characteristics have been assigned a level, i.e., when remaining_characteristics becomes empty

    assigned_this_round = set()     #set() creates an empty set called assigned_this_round to keep track of characteristics that are assigned a level in the current iteration of the while loop

    # Only characteristics assigned BEFORE this level are available
    previously_assigned = set(dependency_levels.keys())
    
    for characteristic in remaining_characteristics:

        # Find all direct parents of this characteristic
        parents = set(
            characteristic_dependencies.loc[    #loc means label-based indexing, so this line selects the rows in the characteristic_dependencies DataFrame where the TO_CHARACTERISTIC column matches the current characteristic and returns the values in the FROM_CHARACTERISTIC column
                characteristic_dependencies["TO_CHARACTERISTIC"] == characteristic,
                "FROM_CHARACTERISTIC"
            ]   # Select the rows where the TO_CHARACTERISTIC column matches the current characteristic and return the values in the FROM_CHARACTERISTIC column
        )

        # Assign the characteristic only if all its parents
        # have already been assigned to an earlier level
        if parents.issubset(previously_assigned):
            dependency_levels[characteristic] = current_level
            assigned_this_round.add(characteristic)

    # Safety check: prevents an infinite loop if the dependency
    # graph contains a cycle
    if not assigned_this_round:
        raise ValueError(
            "Dependency graph may contain a cycle or unresolved dependency."
        )

    remaining_characteristics -= assigned_this_round #-= means set difference, so this line removes the characteristics that have been assigned a level in this round from the remaining_characteristics set
    current_level += 1 #+= means increment, so this line increments the current_level variable by 1 for the next iteration of the while loop

dependency_levels

{'WIDTH_CM': 0,
 'KNOB_TYPE': 0,
 'PANEL_COLOR': 0,
 'OVEN_FUNCTIONS': 0,
 'HANDLE_TYPE': 0,
 'KNOB_COLOR': 1,
 'PIZZA_TRAY': 1,
 'TURNSPIT': 1,
 'KNOB_STRUCTURE': 1,
 'NUMBER_OF_FUNCTIONS': 1,
 'OVEN_TYPE': 1,
 'OVEN_VOLUME_L': 1,
 'HANDLE_COLOR': 1,
 'OVEN_CONTROL': 1,
 'CLEANING': 1,
 'ENERGY_CLASS': 2,
 'PANEL_TYPE': 2,
 'WIFI': 2}

### Inspect Multi-Level Dependencies

The dependency levels are inspected together with their parent characteristics
to understand why each dependent characteristic is assigned to a specific
generation level.

In [23]:
# Create a readable dependency-level summary

dependency_level_rows = []

for characteristic, level in dependency_levels.items():

    parents = sorted(
        characteristic_dependencies.loc[
            characteristic_dependencies["TO_CHARACTERISTIC"] == characteristic,
            "FROM_CHARACTERISTIC"
        ].unique()
    )

    dependency_level_rows.append(
        {
            "CHARACTERISTIC": characteristic,
            "LEVEL": level,
            "PARENTS": ", ".join(parents) if parents else "-"
        }
    )

dependency_level_df = pd.DataFrame(dependency_level_rows)

dependency_level_df = dependency_level_df.sort_values(
    ["LEVEL", "CHARACTERISTIC"]
).reset_index(drop=True)

dependency_level_df

,CHARACTERISTIC,LEVEL,PARENTS
0,HANDLE_TYPE,0,-
1,KNOB_TYPE,0,-
2,OVEN_FUNCTIONS,0,-
3,PANEL_COLOR,0,-
4,WIDTH_CM,0,-
5,CLEANING,1,OVEN_FUNCTIONS
6,HANDLE_COLOR,1,"HANDLE_TYPE, PANEL_COLOR"
7,KNOB_COLOR,1,"KNOB_TYPE, PANEL_COLOR"
8,KNOB_STRUCTURE,1,KNOB_TYPE
9,NUMBER_OF_FUNCTIONS,1,OVEN_FUNCTIONS


### Add Independent Characteristics to the Generation Plan

Characteristics that do not participate in the dependency graph can be
generated independently.

These characteristics are added to Level 0 because they do not require
another characteristic to be generated first.

In [24]:
# All characteristics defined in the master dictionary
all_characteristics = set(
    characteristic_dictionary["CHARACTERISTIC"]
)

# Characteristics participating anywhere in the dependency graph
graph_characteristics = (
    parent_characteristics | child_characteristics #| means set union, so this line combines the parent_characteristics and child_characteristics sets to create a new set containing all characteristics that are either parents or children in the dependency graph
)

# Characteristics outside the dependency graph
independent_characteristics = (
    all_characteristics - graph_characteristics # - means set difference, so this line calculates the set of characteristics that are in all_characteristics but not in graph_characteristics
)


print("Independent characteristics:")
print(sorted(independent_characteristics))

Independent characteristics:
['BAKING_TRAY', 'PACKAGING_TYPE', 'POWER_CABLE', 'ROASTING_TRAY', 'TELESCOPIC_RAILS', 'WIRE_SHELF']


## Final Characteristic Generation Plan

The dependency graph determines the generation order of dependent characteristics.

Characteristics outside the dependency graph are generated independently or
sampled before incompatibility validation.

The final generation plan combines:

- characteristic role,
- inferred generation role,
- dependency level,
- parent characteristics.

In [25]:
# Create the final generation plan from the inferred role matrix

generation_plan = role_matrix.copy()

generation_plan["LEVEL"] = generation_plan["CHARACTERISTIC"].map(
    dependency_levels
)

generation_plan["PARENTS"] = generation_plan["CHARACTERISTIC"].apply(
    lambda characteristic: ", ".join(
        sorted(
            characteristic_dependencies.loc[
                characteristic_dependencies["TO_CHARACTERISTIC"] == characteristic,
                "FROM_CHARACTERISTIC"
            ].unique()
        )
    )
)

generation_plan

,CHARACTERISTIC,IS_CONDITION,IS_ACTION,IS_INCOMPATIBILITY,ROLE,INFERRED_GENERATION_ROLE,LEVEL,PARENTS
0,BAKING_TRAY,False,False,True,INCOMPATIBILITY,VALIDATION_ONLY,NaN,
1,CLEANING,False,True,False,ACTION,DERIVED,1.0,OVEN_FUNCTIONS
2,ENERGY_CLASS,False,True,False,ACTION,DERIVED,2.0,"OVEN_CONTROL, OVEN_FUNCTIONS"
3,HANDLE_COLOR,False,True,True,ACTION + INCOMPATIBILITY,DERIVED,1.0,"HANDLE_TYPE, PANEL_COLOR"
4,HANDLE_TYPE,True,False,False,CONDITION,DRIVER,0.0,
5,KNOB_COLOR,False,True,True,ACTION + INCOMPATIBILITY,DERIVED,1.0,"KNOB_TYPE, PANEL_COLOR"
6,KNOB_STRUCTURE,False,True,False,ACTION,DERIVED,1.0,KNOB_TYPE
7,KNOB_TYPE,True,False,False,CONDITION,DRIVER,0.0,
8,NUMBER_OF_FUNCTIONS,False,True,False,ACTION,DERIVED,1.0,OVEN_FUNCTIONS
9,OVEN_CONTROL,True,True,False,CONDITION + ACTION,INTERMEDIATE,1.0,OVEN_FUNCTIONS


## Define Sampling Strategy

Synthetic products should not be generated using uniform random sampling.

Driver and independent characteristics are sampled using controlled probability
distributions to create a realistic but sufficiently diverse synthetic product portfolio.

The sampling strategy should:

- represent common product configurations more frequently,
- preserve less common alternatives,
- provide sufficient coverage for downstream analysis,
- avoid unrealistic uniform distributions.

In [26]:
# Define sampling weights for WIDTH_CM

sampling_weights = {
    "WIDTH_CM": {
        "V001": 0.10,
        "V002": 0.65,
        "V003": 0.10,
        "V004": 0.15
    }
}

# Validate that probabilities sum to 1

assert np.isclose(
    sum(sampling_weights["WIDTH_CM"].values()),
    1.0
), "WIDTH_CM sampling probabilities must sum to 1."

# Display sampling strategy together with master-data definitions

width_sampling = characteristic_values[
    characteristic_values["CHARACTERISTIC"] == "WIDTH_CM"
][["CHARACTERISTIC", "VALUE", "DEFINITION"]].copy()

width_sampling["PROBABILITY"] = width_sampling["VALUE"].map(
    sampling_weights["WIDTH_CM"]
)

width_sampling

,CHARACTERISTIC,VALUE,DEFINITION,PROBABILITY
0,WIDTH_CM,V001,45,0.10
1,WIDTH_CM,V002,60,0.65
2,WIDTH_CM,V003,70,0.10
3,WIDTH_CM,V004,90,0.15


In [27]:
# Prepare oven function values and create readable function-set IDs

# 1. Get OVEN_FUNCTIONS values from master data
oven_function_values = characteristic_values.loc[
    characteristic_values["CHARACTERISTIC"] == "OVEN_FUNCTIONS",
    ["VALUE", "DEFINITION"]
].copy()

# 2. Create an overview table
oven_function_overview = oven_function_values.copy()

# 3. Extract the function set ID (e.g. "01" from "01-Top+Bottom...")
oven_function_overview["FUNCTION_SET"] = (
    oven_function_overview["DEFINITION"]
    .str.split("-")
    .str[0]
)

# 4. Display the result
oven_function_overview[
    ["VALUE", "FUNCTION_SET", "DEFINITION"]
]

,VALUE,FUNCTION_SET,DEFINITION
30,V001,01,01-Top+Bottom | Bottom | Top
31,V002,02,02-Defrost | Top+Bottom | Top+Bottom+Fan | Gri...
32,V003,03,03-Bottom | Top | Top+Bottom | Top+Bottom+Fan ...
33,V004,04,04-OvenLight | Defrost | Top+Bottom | Top+Bott...
34,V005,05,05-OvenLight | Defrost | Top+Bottom | Top+Bott...
35,V006,06,06-Defrost | Fan+Ring | Top+Bottom | Top+Bott...
36,V007,07,07-Defrost | Fan+Ring | Top+Bottom | Top+Bott...
37,V008,08,08-OvenLight | Defrost | Fan+Ring | Top+Bottom...
38,V009,09,09-Defrost | TURBO | Bottom | Top+Bottom | Top...
39,V010,10,10-OvenLight | Defrost | Fan+Ring | Top+Bottom...


In [28]:
# Infer oven technology type from the function-set definition

def infer_function_type(definition):    #define a function to infer the oven technology type based on the definition of the function set. The function takes a string input (definition) and returns a string output representing the inferred function type. The function first converts the input definition to uppercase to ensure case-insensitive matching. It then checks for specific keywords in the definition to determine the appropriate function type. If "FAN+RING" is found, it returns "MULTIFUNCTION". If either "FAN" or "TURBO" is found, it returns "ELECTROTURBO". If none of these keywords are present, it defaults to returning "STATIC".
    definition_upper = definition.upper()

    if "FAN+RING" in definition_upper:
        return "MULTIFUNCTION"
    elif "FAN" in definition_upper or "TURBO" in definition_upper:
        return "ELECTROTURBO"
    else:
        return "STATIC"


oven_function_overview["FUNCTION_TYPE"] = (
    oven_function_overview["DEFINITION"].apply(infer_function_type)
)

oven_function_overview[
    ["VALUE", "FUNCTION_SET", "FUNCTION_TYPE", "DEFINITION"]
]

,VALUE,FUNCTION_SET,FUNCTION_TYPE,DEFINITION
30,V001,01,STATIC,01-Top+Bottom | Bottom | Top
31,V002,02,ELECTROTURBO,02-Defrost | Top+Bottom | Top+Bottom+Fan | Gri...
32,V003,03,ELECTROTURBO,03-Bottom | Top | Top+Bottom | Top+Bottom+Fan ...
33,V004,04,ELECTROTURBO,04-OvenLight | Defrost | Top+Bottom | Top+Bott...
34,V005,05,ELECTROTURBO,05-OvenLight | Defrost | Top+Bottom | Top+Bott...
35,V006,06,MULTIFUNCTION,06-Defrost | Fan+Ring | Top+Bottom | Top+Bott...
36,V007,07,MULTIFUNCTION,07-Defrost | Fan+Ring | Top+Bottom | Top+Bott...
37,V008,08,MULTIFUNCTION,08-OvenLight | Defrost | Fan+Ring | Top+Bottom...
38,V009,09,ELECTROTURBO,09-Defrost | TURBO | Bottom | Top+Bottom | Top...
39,V010,10,MULTIFUNCTION,10-OvenLight | Defrost | Fan+Ring | Top+Bottom...


In [29]:
# Find RuleIds where OVEN_FUNCTIONS controls NUMBER_OF_FUNCTIONS

function_count_rule_ids = []

for rule_id, group in configuration_rules.groupby("RuleId"):

    has_oven_functions_rule = (
        (group["CHARACTERISTIC"] == "OVEN_FUNCTIONS")
        & (group["Operator"] == "Rule")
    ).any()

    has_function_count_action = (
        (group["CHARACTERISTIC"] == "NUMBER_OF_FUNCTIONS")
        & (group["Operator"] == "Action")
    ).any()

    if has_oven_functions_rule and has_function_count_action:
        function_count_rule_ids.append(rule_id)

function_count_rule_ids

[5, 6, 7, 8, 9]

In [30]:
# Build OVEN_FUNCTIONS -> NUMBER_OF_FUNCTIONS mapping from configuration rules

function_count_mapping = []

for rule_id in function_count_rule_ids:

    group = configuration_rules[
        configuration_rules["RuleId"] == rule_id
    ]

    # Get OVEN_FUNCTIONS values used as conditions
    oven_function_ids = group.loc[
        (group["CHARACTERISTIC"] == "OVEN_FUNCTIONS")
        & (group["Operator"] == "Rule"),
        "VALUE"
    ].tolist()

    # Get the NUMBER_OF_FUNCTIONS value selected by the action
    function_count_value_id = group.loc[
        (group["CHARACTERISTIC"] == "NUMBER_OF_FUNCTIONS")
        & (group["Operator"] == "Action"),
        "VALUE"
    ].iloc[0]

    # Convert the master-data VALUE ID into its readable definition
    function_count = characteristic_values.loc[
        (characteristic_values["CHARACTERISTIC"] == "NUMBER_OF_FUNCTIONS")
        & (characteristic_values["VALUE"] == function_count_value_id),
        "DEFINITION"
    ].iloc[0]

    # Create one mapping row for each OVEN_FUNCTIONS value
    for oven_function_id in oven_function_ids:
        function_count_mapping.append({
            "VALUE": oven_function_id,
            "FUNCTION_COUNT": function_count
        })


function_count_mapping = pd.DataFrame(function_count_mapping)

function_count_mapping

,VALUE,FUNCTION_COUNT
0,V001,3
1,V002,5
2,V003,5
3,V004,8
4,V005,8
5,V006,8
6,V007,8
7,V008,10
8,V009,10
9,V010,11


In [31]:

# Define controlled sampling strategy for OVEN_FUNCTIONS

# 1. Define sampling probabilities using master-data VALUE IDs
oven_function_probabilities = {
    "V001": 0.04,
    "V002": 0.07,
    "V003": 0.07,
    "V004": 0.10,
    "V005": 0.12,
    "V006": 0.12,
    "V007": 0.10,
    "V008": 0.10,
    "V009": 0.08,
    "V010": 0.08,
    "V011": 0.06,
    "V012": 0.06,
}

# 2. Combine function definitions with derived function counts
oven_function_sampling = oven_function_overview.merge(
    function_count_mapping,
    on="VALUE",
    how="left",
    validate="one_to_one"
)

# 3. Create human-readable labels
oven_function_sampling["LABEL"] = (
    oven_function_sampling["FUNCTION_SET"]
    + "_"
    + oven_function_sampling["FUNCTION_COUNT"].astype(str)
    + "_"
    + oven_function_sampling["FUNCTION_TYPE"]
)

# 4. Add sampling probabilities
oven_function_sampling["PROBABILITY"] = (
    oven_function_sampling["VALUE"].map(
        oven_function_probabilities
    )
)

# 5. Validate the sampling strategy
assert oven_function_sampling["FUNCTION_COUNT"].notna().all(), (
    "Missing function count."
)

assert oven_function_sampling["PROBABILITY"].notna().all(), (
    "Missing sampling probability."
)

assert set(oven_function_sampling["VALUE"]) == set(
    oven_function_probabilities
), "Probability IDs do not match master-data IDs."

assert (oven_function_sampling["PROBABILITY"] >= 0).all(), (
    "Probabilities cannot be negative."
)

assert np.isclose(
    oven_function_sampling["PROBABILITY"].sum(),
    1.0
), "Total probability must equal 1."

# 6. Calculate probability distribution by technology type
technology_distribution = (
    oven_function_sampling
    .groupby("FUNCTION_TYPE", as_index=False)["PROBABILITY"]
    .sum()
)

# 7. Display the readable sampling strategy
print("OVEN_FUNCTIONS Sampling Strategy:")

display(
    oven_function_sampling[
        [
            "VALUE",
            "LABEL",
            "FUNCTION_COUNT",
            "FUNCTION_TYPE",
            "PROBABILITY"
        ]
    ]
)

print("\nTechnology Distribution:")
display(technology_distribution)


OVEN_FUNCTIONS Sampling Strategy:


,VALUE,LABEL,FUNCTION_COUNT,FUNCTION_TYPE,PROBABILITY
0,V001,01_3_STATIC,3,STATIC,0.04
1,V002,02_5_ELECTROTURBO,5,ELECTROTURBO,0.07
2,V003,03_5_ELECTROTURBO,5,ELECTROTURBO,0.07
3,V004,04_8_ELECTROTURBO,8,ELECTROTURBO,0.10
4,V005,05_8_ELECTROTURBO,8,ELECTROTURBO,0.12
5,V006,06_8_MULTIFUNCTION,8,MULTIFUNCTION,0.12
6,V007,07_8_MULTIFUNCTION,8,MULTIFUNCTION,0.10
7,V008,08_10_MULTIFUNCTION,10,MULTIFUNCTION,0.10
8,V009,09_10_ELECTROTURBO,10,ELECTROTURBO,0.08
9,V010,10_11_MULTIFUNCTION,11,MULTIFUNCTION,0.08



Technology Distribution:


,FUNCTION_TYPE,PROBABILITY
0,ELECTROTURBO,0.56
1,MULTIFUNCTION,0.40
2,STATIC,0.04


### Variable Inventory — Key Data Structures

This section documents the key variables created by the preceding cells. It is a reference, not an additional data-generation step.

| Variable | Purpose |
|---|---|
| `characteristic_dictionary` | Characteristic definitions from the Excel master |
| `characteristic_values` | Permitted value IDs and readable definitions |
| `configuration_rules` | Configuration rule rows from the Excel master |
| `rule_summary` | Summary of rules grouped by RuleId |
| `role_matrix` | Inferred characteristic roles |
| `characteristic_dependencies` | Unique parent-child dependency pairs |
| `dependency_levels` | Computed level per characteristic |
| `generation_plan` | Role, level and parent summary |
| `sampling_weights` | Sampling probability dictionaries, currently `WIDTH_CM` |
| `width_sampling` | Width definitions joined with probabilities |
| `oven_function_overview` | Function-set IDs, definitions and technology type |
| `function_count_mapping` | Rule-derived function counts per function-set ID |
| `oven_function_probabilities` | Probabilities for 12 oven function sets |
| `oven_function_sampling` | Human-readable function sampling strategy |
| `technology_distribution` | Aggregate probabilities by technology type |
| `product_candidates` | First-stage generated product candidates (created below) |

**Naming convention:** use `sampling_weights["WIDTH_CM"]`, not a separate `width_probabilities` variable.


## Generate Synthetic Product Portfolio

### Step 1 — Sample Primary Driver Characteristics

Generate 500 synthetic product candidates using controlled probability distributions for WIDTH_CM and OVEN_FUNCTIONS.

In [32]:

import numpy as np
import pandas as pd

# Reproducible random number generator
rng = np.random.default_rng(seed=42)        # seed=42 ensures that the random number generator produces the same sequence of random numbers each time the code is run, which is useful for reproducibility in experiments and simulations

# Number of synthetic products
n_products = 500

# Get width value IDs and their probabilities
width_ids = list(sampling_weights["WIDTH_CM"].keys())
width_weights = list(sampling_weights["WIDTH_CM"].values())

# Get oven function IDs and their probabilities
function_ids = list(oven_function_probabilities.keys())
function_weights = list(oven_function_probabilities.values())

# Generate primary driver characteristics
product_candidates = pd.DataFrame({
    "PRODUCT_ID": [
        f"P{i:04d}" for i in range(1, n_products + 1)
    ],
    "WIDTH_CM": rng.choice(
        width_ids,
        size=n_products,
        p=width_weights
    ),
    "OVEN_FUNCTIONS": rng.choice(
        function_ids,
        size=n_products,
        p=function_weights
    )
})

# Display the first 10 product candidates
product_candidates.head(10)


,PRODUCT_ID,WIDTH_CM,OVEN_FUNCTIONS
0,P0001,V003,V009
1,P0002,V002,V011
2,P0003,V004,V011
3,P0004,V002,V006
4,P0005,V001,V007
5,P0006,V004,V009
6,P0007,V003,V005
7,P0008,V003,V010
8,P0009,V002,V006
9,P0010,V002,V003


In [33]:
# Inspect master-data values for width and oven volume

volume_master = characteristic_values[
    characteristic_values["CHARACTERISTIC"].isin(
        ["WIDTH_CM", "OVEN_VOLUME_L"]
    )
][["CHARACTERISTIC", "VALUE", "DEFINITION"]].copy()

display(volume_master)

,CHARACTERISTIC,VALUE,DEFINITION
0,WIDTH_CM,V001,45
1,WIDTH_CM,V002,60
2,WIDTH_CM,V003,70
3,WIDTH_CM,V004,90
76,OVEN_VOLUME_L,V001,50
77,OVEN_VOLUME_L,V002,65
78,OVEN_VOLUME_L,V003,75
79,OVEN_VOLUME_L,V004,82


### Step 2 — Build a Master-Data-Driven Rule Engine

Group configuration rules by RuleId and inspect
their conditions and actions before applying them
to synthetic products.

In [35]:

# Step 2.1 — Group configuration rules by RuleId

# Separate conditions, actions and incompatibilities
rule_conditions = configuration_rules[
    configuration_rules["Operator"] == "Rule"
].copy()

rule_actions = configuration_rules[
    configuration_rules["Operator"] == "Action"
].copy()

# Inspect the structure
print("Condition rows:", len(rule_conditions))
print("Action rows:", len(rule_actions))

print("\nAction types:")
display(rule_actions["Type"].value_counts())

print("\nRule 23:")
display(configuration_rules[
    configuration_rules["RuleId"] == 23
])

Condition rows: 116
Action rows: 43

Action types:


Type
Enable         21
Auto Select    20
Disable         2
Name: count, dtype: int64


Rule 23:


,RuleId,Type,Operator,CHARACTERISTIC,VALUE,DEFINITION
99,23,Disable,Action,KNOB_COLOR,V002,WHITE
100,23,Disable,Action,HANDLE_COLOR,V002,WHITE
101,23,AND,Rule,PANEL_COLOR,V005,DARK SILVER
102,23,AND,Rule,PANEL_COLOR,V004,STAINLESS STEEL
103,23,AND,Rule,PANEL_COLOR,V003,SILVER


In [36]:

# Step 2.2 — Inspect oven volume configuration rules

volume_rule_ids = rule_actions.loc[
    rule_actions["CHARACTERISTIC"] == "OVEN_VOLUME_L",
    "RuleId"
].unique()

volume_rules = configuration_rules[
    configuration_rules["RuleId"].isin(volume_rule_ids)
].sort_values(["RuleId", "Operator"])

print("Volume rule IDs:", volume_rule_ids)
display(volume_rules)


Volume rule IDs: [31 32 33]


,RuleId,Type,Operator,CHARACTERISTIC,VALUE,DEFINITION
151,31,Auto Select,Action,OVEN_VOLUME_L,V001,50
152,31,AND,Rule,WIDTH_CM,V001,45
153,32,Enable,Action,OVEN_VOLUME_L,V002,65
154,32,Enable,Action,OVEN_VOLUME_L,V003,75
155,32,AND,Rule,WIDTH_CM,V002,60
156,32,AND,Rule,WIDTH_CM,V003,70
157,33,Auto Select,Action,OVEN_VOLUME_L,V004,82
158,33,AND,Rule,WIDTH_CM,V004,90


In [42]:

# Step 2.3 — Test master-data-driven rule engine


# Make the project root available for imports
from pathlib import Path
import sys

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)

import importlib
import src.rule_engine

importlib.reload(src.rule_engine)
from src.rule_engine import RuleEngine
from src.rule_engine import RuleEngine

engine = RuleEngine(
    characteristic_values,
    configuration_rules
)

for width in ["V001", "V002", "V003", "V004"]:
    product = {"WIDTH_CM": width}

    options, automatic = engine.get_options(
        "OVEN_VOLUME_L", product
    )

    print(
        f"WIDTH_CM={width} | "
        f"Allowed volumes={options} | "
        f"Auto Select={automatic}"
    )


Project root: /home/benal/oven-product-matching
WIDTH_CM=V001 | Allowed volumes=['V001'] | Auto Select=V001
WIDTH_CM=V002 | Allowed volumes=['V002', 'V003'] | Auto Select=None
WIDTH_CM=V003 | Allowed volumes=['V002', 'V003'] | Auto Select=None


WIDTH_CM=V004 | Allowed volumes=['V004'] | Auto Select=V004


In [44]:

# Step 2.4 — Benchmark the optimized Rule Engine

import importlib
import src.rule_engine
import time

importlib.reload(src.rule_engine)
from src.rule_engine import RuleEngine

engine = RuleEngine(
    characteristic_values,
    configuration_rules
)

# Independent random generator for reproducible benchmarking
test_rng = np.random.default_rng(42)

start_time = time.perf_counter()

product_candidates["OVEN_VOLUME_L"] = [
    engine.choose_value(
        characteristic="OVEN_VOLUME_L",
        product={"WIDTH_CM": width},
        rng=test_rng
    )
    for width in product_candidates["WIDTH_CM"]
]

elapsed = time.perf_counter() - start_time

print(f"Processing time: {elapsed:.3f} seconds")
print(
    "Assigned products:",
    product_candidates["OVEN_VOLUME_L"].notna().sum()
)

display(
    product_candidates[
        ["PRODUCT_ID", "WIDTH_CM", "OVEN_VOLUME_L"]
    ].head(10)
)


Processing time: 0.346 seconds
Assigned products: 500


,PRODUCT_ID,WIDTH_CM,OVEN_VOLUME_L
0,P0001,V003,V002
1,P0002,V002,V003
2,P0003,V004,V004
3,P0004,V002,V003
4,P0005,V001,V001
5,P0006,V004,V004
6,P0007,V003,V002
7,P0008,V003,V002
8,P0009,V002,V003
9,P0010,V002,V002


In [48]:

# Step 2.5 — Determine characteristic generation order

generation_order = (
    generation_plan
    .sort_values(["LEVEL", "CHARACTERISTIC"], na_position="last")
    [["CHARACTERISTIC", "INFERRED_GENERATION_ROLE", "LEVEL", "PARENTS"]]
    .reset_index(drop=True)
)

display(generation_order)

,CHARACTERISTIC,INFERRED_GENERATION_ROLE,LEVEL,PARENTS
0,HANDLE_TYPE,DRIVER,0.0,
1,KNOB_TYPE,DRIVER,0.0,
2,OVEN_FUNCTIONS,DRIVER,0.0,
3,PANEL_COLOR,DRIVER,0.0,
4,WIDTH_CM,DRIVER,0.0,
5,CLEANING,DERIVED,1.0,OVEN_FUNCTIONS
6,HANDLE_COLOR,DERIVED,1.0,"HANDLE_TYPE, PANEL_COLOR"
7,KNOB_COLOR,DERIVED,1.0,"KNOB_TYPE, PANEL_COLOR"
8,KNOB_STRUCTURE,DERIVED,1.0,KNOB_TYPE
9,NUMBER_OF_FUNCTIONS,DERIVED,1.0,OVEN_FUNCTIONS


### Step 3 — Generate Synthetic Product Portfolio

**Objective:** Generate a complete synthetic oven product portfolio using the master-data-driven Rule Engine.

The generation process starts with 500 product candidates and assigns the remaining characteristic values according to the dependency structure defined in `generation_plan`.

The Rule Engine applies configuration rules from the Excel master data, while sampling probabilities control the distribution of selected product characteristics.

**Workflow:**

1. **Step 3.1 — Single Product Test:** Generate and validate one complete product configuration.
2. **Step 3.2 — Full Portfolio Generation:** Apply the algorithm to all 500 product candidates.
3. **Step 3.3 — Quality Validation:** Check completeness, valid values, configuration rules, incompatibilities (INC), and duplicate configurations.
4. **Step 3.4 — Data Export:** Save the validated portfolio as `PRODUCT_MASTER` for subsequent analysis and machine learning tasks.

**Expected output:** A synthetic dataset containing 500 valid oven products, each described by 24 configuration characteristics.

#### Step 3.1 — Generate and Validate a Single Product

**Objective:** Test the complete product generation algorithm on one product before generating the full portfolio.

The algorithm starts with an existing product candidate and assigns missing characteristic values according to the dependency order defined in `generation_plan`.

For each characteristic, the Rule Engine evaluates the applicable configuration rules:

- **Auto Select:** Automatically assigns a required value.
- **Enable:** Restricts the available options to permitted values.
- **Disable:** Removes prohibited values from the available options.
- **INC:** Prevents incompatible characteristic combinations.

The algorithm retries generation when a valid configuration cannot be completed.

**Expected result:** One product containing all 24 characteristics, with no validation errors.

This step is a functional test of the generation process before applying it to the complete dataset.

In [51]:

# Step 3.1 — Generate complete synthetic products

generation_sequence = generation_order["CHARACTERISTIC"].tolist()

# Reproducible generation
generation_rng = np.random.default_rng(42)

def generate_product(base_product, max_attempts=100):
    for attempt in range(max_attempts):
        product = base_product.copy()

        try:
            for characteristic in generation_sequence:
                if characteristic in product:
                    continue

                weights = sampling_weights.get(characteristic)

                product[characteristic] = engine.choose_value(
                    characteristic=characteristic,
                    product=product,
                    rng=generation_rng,
                    weights=weights
                )

            errors = engine.validate_product(product)

            if not errors:
                return product

        except ValueError:
            continue

    raise RuntimeError(
        f"Could not generate a valid product after "
        f"{max_attempts} attempts: {base_product}"
    )

# Test only the first product
first_candidate = product_candidates.iloc[0].to_dict()

first_product = generate_product(first_candidate)

print("Generated characteristics:", len(first_product))
print("Validation errors:", engine.validate_product(first_product))
display(pd.DataFrame([first_product]).T)


Generated characteristics: 25
Validation errors: []


,0
PRODUCT_ID,P0001
WIDTH_CM,V003
OVEN_FUNCTIONS,V009
OVEN_VOLUME_L,V002
HANDLE_TYPE,V001
KNOB_TYPE,V004
PANEL_COLOR,V004
CLEANING,V004
HANDLE_COLOR,V003
KNOB_COLOR,V001


#### Step 3.2 — Generate the Complete Synthetic Product Portfolio

**Objective:** Generate a synthetic portfolio of 500 oven products using the validated Rule Engine.

The algorithm applies the same generation process tested in Step 3.1 to all product candidates.

Each product is completed using:

- Existing driver characteristics and previously assigned values
- Configuration rules loaded from the Excel master data
- Sampling probabilities where defined
- Random selection among valid options when no specific probabilities are provided
- Rule validation and retry logic for invalid configurations

After generation, the dataset is checked for:

- Total product count
- Number of assigned characteristics
- Missing values
- Duplicate product configurations

**Expected result:** A complete portfolio of 500 synthetic oven products with 24 characteristic values per product.

The complete portfolio will undergo an additional quality-control step before being exported as the final `PRODUCT_MASTER` dataset.

In [52]:

# Step 3.2 — Generate the complete product portfolio

import time

start_time = time.perf_counter()

generated_products = []

for _, row in product_candidates.iterrows():
    base_product = row.to_dict()
    generated_products.append(generate_product(base_product))

product_master = pd.DataFrame(generated_products)

elapsed = time.perf_counter() - start_time

# Product ID is an identifier, not a configuration characteristic
characteristic_columns = [
    col for col in product_master.columns
    if col in engine.all_options
]

duplicate_count = product_master.duplicated(
    subset=characteristic_columns
).sum()

print(f"Generation time: {elapsed:.2f} seconds")
print(f"Total products: {len(product_master)}")
print(f"Characteristics: {len(characteristic_columns)}")
print(f"Duplicate configurations: {duplicate_count}")
print(f"Missing values: {product_master[characteristic_columns].isna().sum().sum()}")

display(product_master.head())


Generation time: 3.60 seconds
Total products: 500
Characteristics: 24
Duplicate configurations: 0
Missing values: 0


,PRODUCT_ID,WIDTH_CM,OVEN_FUNCTIONS,OVEN_VOLUME_L,HANDLE_TYPE,KNOB_TYPE,PANEL_COLOR,CLEANING,HANDLE_COLOR,KNOB_COLOR,...,TURNSPIT,ENERGY_CLASS,PANEL_TYPE,WIFI,BAKING_TRAY,PACKAGING_TYPE,POWER_CABLE,ROASTING_TRAY,TELESCOPIC_RAILS,WIRE_SHELF
0,P0001,V003,V009,V002,V004,V004,V004,V004,V003,V003,...,V002,V001,V001,V001,V003,V001,V003,V002,V001,V003
1,P0002,V002,V011,V003,V004,V004,V003,V004,V003,V003,...,V002,V001,V001,V001,V002,V001,V001,V002,V003,V001
2,P0003,V004,V011,V004,V005,V005,V002,V004,V002,V001,...,V002,V002,V001,V001,V003,V001,V001,V003,V002,V002
3,P0004,V002,V006,V003,V004,V004,V004,V003,V001,V001,...,V002,V001,V001,V002,V002,V001,V004,V003,V003,V003
4,P0005,V001,V007,V001,V002,V005,V003,V003,V001,V003,...,V001,V001,V001,V002,V003,V001,V003,V001,V003,V002


#### Step 3.3 — Validate Product Portfolio Quality

**Objective:** Verify that all 500 generated products satisfy the configuration rules and data-quality requirements.

The validation process checks:

- **Completeness:** All 24 characteristics are assigned.
- **Valid Values:** Every characteristic value exists in the master data.
- **Configuration Rules:** Auto Select, Enable and Disable rules are respected.
- **Incompatibility (INC):** No product contains a forbidden combination.
- **Uniqueness:** No duplicate product configurations exist.

**Expected result:** Zero validation errors across the complete product portfolio.

The dataset will only be exported after these checks have passed.

In [53]:

# Step 3.3 — Validate all generated products

validation_results = []

# Read INC combinations directly from Excel rule data
inc_combinations = []

for rule_id, group in configuration_rules.groupby("RuleId"):
    inc_rows = group[group["Type"] == "INC"]

    if not inc_rows.empty:
        combination = dict(
            zip(inc_rows["CHARACTERISTIC"], inc_rows["VALUE"])
        )
        inc_combinations.append((rule_id, combination))

for _, row in product_master.iterrows():
    product = row.to_dict()

    # Validate Auto Select, Enable, Disable and valid values
    errors = engine.validate_product(product)

    # Independently inspect INC combinations
    inc_violations = [
        rule_id
        for rule_id, combination in inc_combinations
        if all(
            product.get(char) == value
            for char, value in combination.items()
        )
    ]

    validation_results.append({
        "PRODUCT_ID": product["PRODUCT_ID"],
        "RULE_ERRORS": len(errors),
        "INC_VIOLATIONS": len(inc_violations),
        "DETAILS": "; ".join(errors),
        "INC_RULE_IDS": inc_violations
    })

validation_report = pd.DataFrame(validation_results)

print("Total products:", len(product_master))
print("Characteristics:", len(characteristic_columns))
print("Missing values:",
      product_master[characteristic_columns].isna().sum().sum())
print("Duplicate configurations:",
      product_master.duplicated(subset=characteristic_columns).sum())
print("Products with rule errors:",
      (validation_report["RULE_ERRORS"] > 0).sum())
print("Products with INC violations:",
      (validation_report["INC_VIOLATIONS"] > 0).sum())

display(
    validation_report[
        (validation_report["RULE_ERRORS"] > 0) |
        (validation_report["INC_VIOLATIONS"] > 0)
    ].head(20)
)


Total products: 500
Characteristics: 24
Missing values: 0
Duplicate configurations: 0
Products with rule errors: 0
Products with INC violations: 0


,PRODUCT_ID,RULE_ERRORS,INC_VIOLATIONS,DETAILS,INC_RULE_IDS


#### Step 3.4 — Export the Validated Product Master

**Objective:** Save the validated synthetic product portfolio for subsequent data analysis, customer request matching, and machine learning.

The exported dataset contains:

- 500 unique synthetic oven products
- 24 configuration characteristics per product
- Characteristic value IDs linked to the Excel master data
- No missing values, duplicate configurations, or detected rule violations

The dataset is exported as `product_master.csv` to the `data/processed/` directory.

This file will serve as the product portfolio reference for the next stages of the project.

In [54]:

# Step 3.4 — Export validated product portfolio

from pathlib import Path

# Stop export if quality checks fail
assert len(product_master) == 500
assert len(characteristic_columns) == 24
assert product_master[characteristic_columns].notna().all().all()
assert duplicate_count == 0
assert (validation_report["RULE_ERRORS"] == 0).all()
assert (validation_report["INC_VIOLATIONS"] == 0).all()

# Create output directory if necessary
output_dir = project_root / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

# Keep identifier and characteristic columns
export_columns = ["PRODUCT_ID"] + characteristic_columns

output_path = output_dir / "product_master.csv"

product_master[export_columns].to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Export completed:", output_path)
print("Exported shape:", product_master[export_columns].shape)


Export completed: /home/benal/oven-product-matching/data/processed/product_master.csv
Exported shape: (500, 25)
